## 01g_ingest_ipps_final_rule — RCM Intelligence Platform
### Purpose
Ingests CMS IPPS (Inpatient Prospective Payment System) Final Rule
Impact File into Bronze Delta tables.

### Data sources
- CMS IPPS Final Rule Impact File (FY2026)
- Published annually by CMS (typically August)
- ZIP archive containing Excel file with hospital-specific payment projections

### What this does
1. Downloads IPPS Final Rule Impact File ZIP from CMS
2. Extracts Excel file from ZIP
3. Reads Excel data with pandas
4. Adds audit metadata columns to every row
5. Writes raw data to Bronze Delta table (overwrite mode)
6. Runs verification and data quality checks

### Outputs
- rcm_platform.rcm_bronze.ipps_final_rule_raw

### Notes
- No transformations here — raw data only
- Annual data — safe to overwrite each fiscal year
- Schema inference is on — new CMS columns handled automatically
- File contains ~3,000 hospitals with 50-100 columns

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Bronze |
| Run order  | 7 — independent of other bronze jobs |
| Depends on | 00_config, 00_utils |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# Every ingestion run gets a unique batch ID
# This lets us trace exactly which run wrote which rows
# ============================================================

import uuid
from datetime import datetime, timezone

BATCH_ID         = str(uuid.uuid4())
BATCH_DATE       = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP  = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME    = "01g_ingest_ipps_final_rule"
FISCAL_YEAR      = "FY2024"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"Batch timestamp : {BATCH_TIMESTAMP}")
print(f"Fiscal year     : {FISCAL_YEAR}")
print(f"Target table    : {BRONZE_DB}.ipps_final_rule_raw")

In [0]:
# ============================================================
# STEP 1 — DOWNLOAD IPPS IMPACT FILE FROM CMS
# Downloads ZIP, extracts Excel file
# ============================================================

import requests
import zipfile
import os

print("=" * 55)
print("  DOWNLOADING CMS IPPS IMPACT FILE")
print("=" * 55)

# CMS IPPS Final Rule FY2024 URL
# Note: Update annually when new fiscal year is published
IPPS_URL = "https://www.cms.gov/files/zip/fy2024-ipps-fr-impact-file.zip"

# Temporary storage paths (use /tmp for serverless compute)
TMP_DIR    = "/tmp/cms_ipps_fy2024/"
ZIP_PATH   = f"{TMP_DIR}ipps_impact.zip"
EXCEL_PATH = f"{TMP_DIR}ipps_impact.xlsx"

# Create temp directory
os.makedirs(TMP_DIR, exist_ok=True)

try:
    print(f"Downloading from: {IPPS_URL}")
    
    # Download ZIP file
    response = requests.get(IPPS_URL, timeout=API_TIMEOUT_SECS, stream=True)
    response.raise_for_status()
    
    with open(ZIP_PATH, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
    print(f"  Downloaded {zip_size_mb:.2f} MB")
    
    # Extract Excel file from ZIP
    print("Extracting Excel file...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        print(f"  Found {len(file_list)} file(s) in ZIP")
        
        # Find Excel file
        excel_files = [f for f in file_list if f.endswith(('.xlsx', '.xls'))]
        
        if not excel_files:
            raise ValueError("No Excel file found in ZIP archive")
        
        # Extract first Excel file
        excel_file = excel_files[0]
        print(f"  Extracting: {excel_file}")
        
        zip_ref.extract(excel_file, TMP_DIR)
        
        # Rename to standard name
        extracted_path = os.path.join(TMP_DIR, excel_file)
        os.rename(extracted_path, EXCEL_PATH)
    
    print(f"  Excel file ready: {EXCEL_PATH}")
    
except Exception as e:
    print(f"  Download/extraction failed: {e}")
    raise

In [0]:
# ============================================================
# STEP 2 — INSTALL OPENPYXL FOR EXCEL READING
# Required for pandas to read .xlsx files
# ============================================================

print("=" * 55)
print("  INSTALLING EXCEL DEPENDENCIES")
print("=" * 55)

%pip install openpyxl --quiet

In [0]:
# ============================================================
# STEP 3 — READ EXCEL AND CONVERT TO SPARK DATAFRAME
# Reads with pandas then converts to Spark
# ============================================================

import pandas as pd
from pyspark.sql.functions import lit, col

print("=" * 55)
print("  READING EXCEL FILE")
print("=" * 55)

try:
    print(f"Reading Excel file: {EXCEL_PATH}")
    
    # First, inspect available sheets
    print("\nInspecting Excel file structure...")
    excel_file = pd.ExcelFile(EXCEL_PATH)
    available_sheets = excel_file.sheet_names
    print(f"Available sheets: {available_sheets}")
    
    # Find the data sheet (not 'Variable Descriptions')
    # FY2024 might use different naming than FY2026
    data_sheet = None
    for sheet in available_sheets:
        if 'variable' not in sheet.lower() and 'description' not in sheet.lower():
            data_sheet = sheet
            break
    
    if not data_sheet:
        # If we can't find it automatically, use the second sheet (index 1)
        data_sheet = available_sheets[1] if len(available_sheets) > 1 else available_sheets[0]
    
    print(f"Using data sheet: '{data_sheet}'")
    print("This may take 2-3 minutes for ~3,000 hospitals...")
    
    # Read Excel with pandas
    # Skip first row (title row) and use row 1 as header
    pdf = pd.read_excel(
        EXCEL_PATH,
        sheet_name=data_sheet,  # Use the detected data sheet
        header=1,                # Row 1 contains column names
        dtype=str                # Preserve all data as string
    )
    
    print(f"  Read {len(pdf):,} rows and {len(pdf.columns)} columns")
    
    # Clean column names
    pdf.columns = [
        c.strip()
         .lower()
         .replace(' ', '_')
         .replace('/', '_')
         .replace('-', '_')
         .replace('(', '')
         .replace(')', '')
         .replace('.', '')
         .replace('__', '_')
         .replace(',', '')
        for c in pdf.columns
    ]
    
    print("\nSample column names (first 10):")
    for c in pdf.columns[:10]:
        print(f"  {c}")
    if len(pdf.columns) > 10:
        print(f"  ... and {len(pdf.columns) - 10} more columns")
    
    # Convert to Spark DataFrame
    print("\nConverting to Spark DataFrame...")
    df_raw = spark.createDataFrame(pdf)
    
    print(f"  Conversion complete")
    
except Exception as e:
    print(f"  Failed to read Excel file: {e}")
    raise

In [0]:
# ============================================================
# STEP 4 — ADD AUDIT METADATA
# Never modify raw values, only append audit columns
# ============================================================

print("=" * 55)
print("  ADDING AUDIT METADATA")
print("=" * 55)

# Add audit metadata columns
df_bronze = df_raw \
    .withColumn("_source",           lit("cms_ipps_final_rule")) \
    .withColumn("_batch_id",         lit(BATCH_ID)) \
    .withColumn("_batch_date",       lit(BATCH_DATE)) \
    .withColumn("_ingested_at",      lit(BATCH_TIMESTAMP).cast("timestamp")) \
    .withColumn("_pipeline_name",    lit(PIPELINE_NAME)) \
    .withColumn("_pipeline_version", lit(PIPELINE_VERSION)) \
    .withColumn("_fiscal_year",      lit(FISCAL_YEAR))

print(f"  Added 7 audit columns")
print(f"  Total columns: {len(df_bronze.columns)}")

In [0]:
# ============================================================
# STEP 5 — WRITE TO BRONZE DELTA TABLE
# Overwrite mode since this is annual data
# ============================================================

print("=" * 55)
print("  WRITING TO BRONZE TABLE")
print("=" * 55)

target_table = f"{BRONZE_DB}.ipps_final_rule_raw"
print(f"Writing to: {target_table}")

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("mergeSchema", "true") \
    .saveAsTable(target_table)

# Add table comment
spark.sql(f"""
    COMMENT ON TABLE {target_table}
    IS 'CMS IPPS Final Rule Impact File ({FISCAL_YEAR}) - Raw hospital-specific payment projections including wage index, DSH, GME impacts, and policy changes. Updated annually. Source: CMS.gov'
""")

row_count = spark.table(target_table).count()
print(f"  Table: {target_table}")
print(f"  Rows written: {row_count:,}")

In [0]:
# ============================================================
# STEP 6 — VERIFICATION
# Confirm table exists and has correct row counts
# ============================================================

print("=" * 55)
print("  BRONZE VERIFICATION")
print("=" * 55)

target_table = f"{BRONZE_DB}.ipps_final_rule_raw"

print(f"\nTable: {target_table}")
spark.sql(f"""
    SELECT
        COUNT(*)                  AS total_rows,
        COUNT(DISTINCT _batch_id) AS batches_loaded,
        MIN(_ingested_at)         AS first_ingested,
        MAX(_ingested_at)         AS last_ingested,
        MAX(_fiscal_year)         AS fiscal_year
    FROM {target_table}
""").show(truncate=False)

# Data quality checks
df_verify = spark.table(target_table)
total_rows = df_verify.count()

print("\nData Quality Checks:")
print(f"  Total hospitals: {total_rows:,}")

if total_rows < 2500:
    print("  ⚠️  Warning: Expected ~3,000 hospitals. Data may be incomplete.")
elif total_rows > 3500:
    print("  ⚠️  Warning: More hospitals than expected. Check for duplicates.")
else:
    print("  ✓ Row count looks good")

print(f"\n  Total columns: {len(df_verify.columns)}")
print("\n  Sample data (first 5 rows):")
df_verify.select(df_verify.columns[:5]).show(5, truncate=False)

In [0]:
# ============================================================
# STEP 7 — CLEANUP TEMPORARY FILES
# Remove downloaded files to save space
# ============================================================

print("=" * 55)
print("  CLEANUP")
print("=" * 55)

import shutil

try:
    if os.path.exists(TMP_DIR):
        shutil.rmtree(TMP_DIR)
        print(f"  Removed temp directory: {TMP_DIR}")
    
except Exception as e:
    print(f"  Cleanup warning: {e}")
    print("  (Not critical - temp files can be cleaned up manually)")

print("\n" + "=" * 55)
print("  INGESTION COMPLETE")
print("=" * 55)
print(f"\nSuccessfully ingested IPPS Final Rule Impact File")
print(f"  Table: {BRONZE_DB}.ipps_final_rule_raw")
print(f"  Records: {total_rows:,}")
print(f"  Fiscal Year: {FISCAL_YEAR}")
print(f"  Batch ID: {BATCH_ID}")